# AgriConflict — InternVL3-8B sur Colab

Complete le quatrieme modele de la grille, le quota GPU Kaggle etant epuise.

**Avant de lancer :** menu *Execution -> Modifier le type d'execution -> GPU (T4)*.

Le corpus est telecharge depuis Kaggle pour garantir des sondes **strictement
identiques** a celles des trois autres modeles (meme seed, memes images).
Duree attendue : environ 1 h 30.


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes kaggle
import torch
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "AUCUN — changez le type d'execution !")

## 1. Identifiants Kaggle

Deposez votre `kaggle.json`. Il sert uniquement a telecharger le corpus de sondes.

In [ ]:
from google.colab import files
import os, pathlib
up = files.upload()                     # choisissez kaggle.json
pathlib.Path("/root/.kaggle").mkdir(exist_ok=True)
pathlib.Path("/root/.kaggle/kaggle.json").write_bytes(next(iter(up.values())))
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("ok")

## 2. Corpus de sondes (identique aux trois autres modeles)

In [ ]:
!kaggle kernels output abdelbassetchenna/agriconflict-01-build -p /content/probes
import json, pathlib
P = pathlib.Path("/content/probes")
probes = [json.loads(l) for l in
          (P / "probes.jsonl").read_text(encoding="utf-8").splitlines() if l]
print(len(probes), "sondes |", len({p["uid"] for p in probes}), "items")
missing = [p for p in probes if not (P / p["image"]).exists()]
print("images manquantes :", len(missing))
assert not missing, "telechargement incomplet — relancez cette cellule"


## 3. Chargement d'InternVL3-8B

`crop_to_patches=False` neutralise le decoupage dynamique en tuiles. Nos images
font 448 px au maximum : une vue unique correspond exactement a ce que voient
Qwen2.5-VL et Qwen2.5-Omni. Sans cela une sonde produit des milliers de tokens
visuels et le run depasse toute limite de session — c'est ce qui a fait annuler
le run Kaggle apres 12 h.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID, SHORT = "OpenGVLab/InternVL3-8B-hf", "internvl3-8b"
ALTS = ["OpenGVLab/InternVL3_5-8B-HF", "OpenGVLab/InternVL3-8B"]
PROC_KWARGS = {"crop_to_patches": False}

qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=torch.float16,
                        bnb_4bit_use_double_quant=True)
model = processor = used_id = None
for mid in [MODEL_ID, *ALTS]:
    try:
        print("tentative :", mid)
        processor = AutoProcessor.from_pretrained(mid, trust_remote_code=True)
        model = AutoModelForImageTextToText.from_pretrained(
            mid, trust_remote_code=True, torch_dtype=torch.float16,
            attn_implementation="sdpa", device_map="auto",
            low_cpu_mem_usage=True, quantization_config=qc)
        model.eval(); used_id = mid
        print("charge :", mid); break
    except Exception as e:
        print("  echec :", type(e).__name__, str(e)[:160])
assert model is not None, "aucun identifiant utilisable"

tok = getattr(processor, "tokenizer", processor)
LET = {}
for L in "ABCDE":
    ids = set()
    for v in (L, " " + L, "\n" + L):
        enc = tok.encode(v, add_special_tokens=False)
        if enc:
            ids.add(enc[0])
    LET[L] = sorted(ids)
print("lettres :", {k: v[0] for k, v in LET.items()})

## 4. Evaluation

Choix multiple force, lecture des logits du premier token : un seul passage avant
par sonde. Ecriture incrementale — relancer la cellule reprend ou elle s'est
arretee si la session est coupee.

In [ ]:
import json, time, pathlib
from PIL import Image

OUT = pathlib.Path("/content/results_" + SHORT + ".jsonl")
done = set()
if OUT.exists():
    done = {json.loads(l)["probe_id"]
            for l in OUT.read_text(encoding="utf-8").splitlines() if l}
    print("reprise :", len(done))
todo = [p for p in probes if p["probe_id"] not in done]
print("a traiter :", len(todo))


@torch.inference_mode()
def score(img, prompt):
    msgs = [{"role": "user", "content": [{"type": "image"},
                                         {"type": "text", "text": prompt}]}]
    txt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    try:
        inp = processor(text=[txt], images=[img], return_tensors="pt", **PROC_KWARGS)
    except TypeError:
        inp = processor(text=[txt], images=[img], return_tensors="pt")
    inp = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inp.items()}
    lg = model(**inp).logits[0, -1, :].float()
    s = {L: max(lg[i].item() for i in ids) for L, ids in LET.items()}
    return max(s, key=s.get), s


t0, nok = time.time(), 0
with OUT.open("a", encoding="utf-8") as fh:
    for i, p in enumerate(todo):
        try:
            with Image.open(P / p["image"]) as im:
                pred, sc = score(im.convert("RGB"), p["prompt"])
            ok = int(pred == p["gold_letter"]); nok += ok
            chose = ("image" if pred == p["letter_image"] else
                     "text" if pred == p["letter_text"] else
                     "abstain" if pred == p["letter_abstain"] else "other")
            fh.write(json.dumps({
                "probe_id": p["probe_id"], "uid": p["uid"], "model": SHORT,
                "model_id": used_id, "lang": p["lang"], "cell": p["cell"],
                "pred": pred, "gold": p["gold_letter"], "correct": ok, "chose": chose,
                "subset": p["subset"], "q_img": p["q_img"], "q_rep": p["q_rep"],
                "img_ok": p["img_ok"], "rep_ok": p["rep_ok"],
                "letter_image": p["letter_image"], "letter_text": p["letter_text"],
                "letter_abstain": p["letter_abstain"],
                "logits": {k: round(v, 4) for k, v in sc.items()},
            }, ensure_ascii=False) + "\n")
        except Exception as e:
            fh.write(json.dumps({"probe_id": p["probe_id"], "model": SHORT,
                                 "cell": p["cell"], "error": repr(e)},
                                ensure_ascii=False) + "\n")
        if (i + 1) % 500 == 0:
            fh.flush()
            r = (time.time() - t0) / (i + 1)
            print("  %d/%d | %.3f s/sonde | acc %.3f | reste ~%.0f min"
                  % (i + 1, len(todo), r, nok / (i + 1), r * (len(todo) - i - 1) / 60))
print("termine en", round((time.time() - t0) / 60, 1), "min")

## 5. Controle rapide, puis telechargement

In [ ]:
import json
rows = [json.loads(l) for l in OUT.read_text(encoding="utf-8").splitlines() if l]
ok = [r for r in rows if "error" not in r]
print("n=%d  erreurs=%d  model_id=%s"
      % (len(rows), len(rows) - len(ok), sorted({r["model_id"] for r in ok})))
for c in ("V0_gate", "T0_gate"):
    v = [r["correct"] for r in ok if r["cell"] == c]
    print("  %-10s acc=%.3f" % (c, sum(v) / len(v)))
sub = {}
for s in ("S_img", "S_txt", "S_none", "S_both"):
    v = [r["correct"] for r in ok if r["subset"] == s and not r["cell"].endswith("_gate")]
    sub[s] = sum(v) / len(v) if v else float("nan")
    print("  %-8s acc=%.3f" % (s, sub[s]))
print("\n  A = %+.3f   (plafond Bayes-provenance = +0.634)"
      % (sub["S_img"] + sub["S_txt"] - 1))

from google.colab import files
files.download(str(OUT))

## 6. Ensuite

Recuperez `results_internvl3-8b.jsonl` et deposez-le dans
`agriconflict/_log_internvl/`. Les scripts d'analyse et SAGA le detecteront
automatiquement et rempliront les cases `[pending]` du §6.